# Read Bronze orders


In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Read raw JSON from Bronze
orders_df = spark.read \
    .option("multiline", "true") \
    .json("abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/raw/orders/orders_20240101_20240131.json")

print("Raw rows:", orders_df.count())
orders_df.printSchema()

StatementMeta(, 46146060-42ac-45a8-a588-e0c4af13fc2d, 4, Finished, Available, Finished, False)

Raw rows: 1400
root
 |-- city: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- discount_code: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- discount_pct: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- item_total: double (nullable = true)
 |    |    |-- product_id: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |    |    |-- unit_price_str: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- state: string (nullable = true)



In [3]:
orders_df.show(4)


StatementMeta(, 46146060-42ac-45a8-a588-e0c4af13fc2d, 7, Finished, Available, Finished, False)

+---------+-----------+-------------+--------------------+-------------------+----------+------------+-----------+----------------+-----+
|     city|customer_id|discount_code|               items|         order_date|  order_id|order_status|order_total|  payment_method|state|
+---------+-----------+-------------+--------------------+-------------------+----------+------------+-----------+----------------+-----+
|    Delhi|  CUST10172|       SAVE24|[{Toys, 0.2, ITEM...|2024-01-01T20:07:01|ORD0000001|   delivered|   42610.12|      Debit Card|   KA|
|   Jaipur|  CUST10327|       SAVE11|[{Books, 0.19, IT...|2024-01-01T17:07:59|ORD0000002|     shipped|  104337.31|          Wallet|   DL|
|Bangalore|  CUST10056|         NULL|[{Beauty, 0.0, IT...|2024-01-01T07:52:02|ORD0000003|   delivered|  153580.04|     Net Banking|   GJ|
|   Jaipur|  CUST10276|         NULL|[{Sports, 0.03, I...|2024-01-01T19:29:33|ORD0000004|     shipped|   92692.71|Cash on Delivery|   MH|
+---------+-----------+-----------

In [4]:
typed_df = orders_df \
    .withColumn(
        "order_timestamp",
        F.to_timestamp(F.col("order_date"), "yyyy-MM-dd'T'HH:mm:ss")
    ) \
    .withColumn("order_date_key", F.to_date(F.col("order_date"), "yyyy-MM-dd'T'HH:mm:ss")) \
    .withColumn("order_amount",   F.col("order_total").cast(T.DecimalType(12,2))) \
    .withColumn("order_hour",     F.hour(F.col("order_timestamp"))) \
    .withColumn("is_weekend",     F.dayofweek(F.col("order_timestamp")).isin([1,7]))

print("After typing:", typed_df.count())

StatementMeta(, 46146060-42ac-45a8-a588-e0c4af13fc2d, 9, Finished, Available, Finished, False)

After typing: 1400


In [6]:
typed_df.printSchema()

StatementMeta(, 46146060-42ac-45a8-a588-e0c4af13fc2d, 11, Finished, Available, Finished, False)

root
 |-- city: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- discount_code: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- discount_pct: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- item_total: double (nullable = true)
 |    |    |-- product_id: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |    |    |-- unit_price_str: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_total: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- order_date_key: date (nullable = true)
 |-- order_amount: decimal(1

In [7]:
# Map messy city values to clean standard names
city_map = {
    "mumbai":"Mumbai","mum":"Mumbai","mmbai":"Mumbai","bombay":"Mumbai",
    "delhi":"Delhi","new delhi":"Delhi","ncr":"Delhi",
    "bangalore":"Bangalore","bengaluru":"Bangalore","blr":"Bangalore",
    "hyderabad":"Hyderabad","hyd":"Hyderabad",
    "chennai":"Chennai","madras":"Chennai",
    "pune":"Pune","kolkata":"Kolkata","calcutta":"Kolkata",
    "ahmedabad":"Ahmedabad","jaipur":"Jaipur","surat":"Surat"
}

def clean_city(city):
    if city is None:
        return "Unknown"
    return city_map.get(city.lower().strip(), city.strip().title())

clean_city_udf = F.udf(clean_city, T.StringType())

standardized_df = typed_df \
    .withColumn("city_clean",    clean_city_udf(F.col("city"))) \
    .withColumn("state_clean",   F.upper(F.trim(F.col("state")))) \
    .withColumn("status_clean",  F.lower(F.trim(F.col("order_status"))))

# Quick check — see city standardization result
standardized_df.groupBy("city","city_clean").count().orderBy(F.desc("count")).show(10)

StatementMeta(, 46146060-42ac-45a8-a588-e0c4af13fc2d, 12, Finished, Available, Finished, False)

+---------+----------+-----+
|     city|city_clean|count|
+---------+----------+-----+
|  Kolkata|   Kolkata|  145|
|Ahmedabad| Ahmedabad|  143|
|    Surat|     Surat|  142|
|  Chennai|   Chennai|  142|
|     Pune|      Pune|  138|
|Bangalore| Bangalore|  136|
|    Delhi|     Delhi|  124|
|Hyderabad| Hyderabad|  122|
|   Mumbai|    Mumbai|  119|
|   Jaipur|    Jaipur|  118|
+---------+----------+-----+
only showing top 10 rows



In [8]:
null_handled_df = standardized_df \
    .withColumn(
        # null discount_code = no promo used — valid business case
        "discount_code_clean",
        F.coalesce(F.col("discount_code"), F.lit("NO_PROMO"))
    ) \
    .withColumn(
        # null customer_id = guest checkout — not an error
        "is_guest_order",
        F.col("customer_id").isNull()
    ) \
    .withColumn(
        "customer_id_clean",
        F.when(F.col("customer_id").isNull(),
               F.concat(F.lit("GUEST_"), F.col("order_id"))
        ).otherwise(F.col("customer_id"))
    ) \
    .withColumn("has_amount_error", F.col("order_amount").isNull())

StatementMeta(, 46146060-42ac-45a8-a588-e0c4af13fc2d, 13, Finished, Available, Finished, False)

In [9]:
# Keep only the latest version of each order_id
window = Window.partitionBy("order_id").orderBy(F.desc("order_date"))

dedup_df = null_handled_df \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

print("After dedup:", dedup_df.count())

StatementMeta(, 46146060-42ac-45a8-a588-e0c4af13fc2d, 14, Finished, Available, Finished, False)

After dedup: 1400


In [17]:
from pyspark.sql import functions as F

# Create Silver Orders DataFrame
silver_orders = dedup_df.select(
    F.col("order_id"),
    F.col("customer_id_clean").alias("customer_id"),
    F.col("is_guest_order"),
    F.col("order_timestamp"),
    F.col("order_date_key").alias("order_date"),
    F.col("order_hour"),
    F.col("is_weekend"),
    F.col("city_clean").alias("city"),
    F.col("state_clean").alias("state"),
    F.col("order_amount"),
    F.col("status_clean").alias("order_status"),
    F.col("payment_method"),
    F.col("discount_code_clean").alias("discount_code"),
    F.size(F.col("items")).alias("item_count"),
    F.col("has_amount_error"),
    F.current_timestamp().alias("silver_created_at")
)

# Target Delta Path (Silver Lakehouse)
TARGET_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/silver_orders"
)

# Write DataFrame as Delta
(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("order_date")
    .save(TARGET_PATH)
)

# Validate the written data
loaded_df = (
    spark.read
    .format("delta")
    .load(TARGET_PATH)
)

row_count = loaded_df.count()

print("=" * 60)
print("✅ Silver Orders written successfully!")
print(f"📂 Path      : {TARGET_PATH}")
print(f"📊 Total Rows: {row_count}")
print("=" * 60)

# Optional: Display sample data
loaded_df.show(10, truncate=False)

StatementMeta(, 46146060-42ac-45a8-a588-e0c4af13fc2d, 22, Finished, Available, Finished, False)

✅ Silver Orders written successfully!
📂 Path      : abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Tables/silver_orders
📊 Total Rows: 1400
+----------+-----------+--------------+-------------------+----------+----------+----------+---------+-----+------------+------------+----------------+-------------+----------+----------------+--------------------------+
|order_id  |customer_id|is_guest_order|order_timestamp    |order_date|order_hour|is_weekend|city     |state|order_amount|order_status|payment_method  |discount_code|item_count|has_amount_error|silver_created_at         |
+----------+-----------+--------------+-------------------+----------+----------+----------+---------+-----+------------+------------+----------------+-------------+----------+----------------+--------------------------+
|ORD0001361|CUST10323  |false         |2024-01-31 17:58:27|2024-01-31|17        |false     |Ahmedabad|KA   |91454.37    |delivered   |UPI             |S

In [22]:
from pyspark.sql import functions as F

TARGET_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/silver_orders"
)

# Read Delta table
silver_df = (
    spark.read
    .format("delta")
    .load(TARGET_PATH)
)

# Create temp view
silver_df.createOrReplaceTempView("silver_orders")

# Run SQL
spark.sql("""
SELECT
    COUNT(*) AS total_orders,
    MIN(order_date) AS from_date,
    MAX(order_date) AS to_date,
    COUNT(DISTINCT city) AS cities,
    ROUND(AVG(order_amount), 2) AS avg_amount,
    SUM(CAST(is_guest_order AS INT)) AS guest_orders
FROM silver_orders
""").show()

StatementMeta(, 46146060-42ac-45a8-a588-e0c4af13fc2d, 27, Finished, Available, Finished, False)

+------------+----------+----------+------+----------+------------+
|total_orders| from_date|   to_date|cities|avg_amount|guest_orders|
+------------+----------+----------+------+----------+------------+
|        1400|2024-01-01|2024-01-31|    10| 140983.49|          27|
+------------+----------+----------+------+----------+------------+

